## 0. This checkout, not whatever is installed

`zou_lab_control_v2` is the one entry: importing it puts this checkout's eight layers
on the path, ahead of anything else.  It has to come **first** -- if a `zlc_*` module
was already imported from somewhere else, it refuses out loud rather than leaving two
copies in one kernel.  (If that happens: restart the kernel and run this cell first.)


In [1]:
import zou_lab_control_v2

import zlc_atom
print('root in use:', zou_lab_control_v2.__file__)
print('tested package:', zlc_atom.__file__)


root in use: C:\Users\eadri\Dropbox\WorkCode\Github\Zou_lab_control_v2\zou_lab_control_v2\__init__.py
tested package: C:\Users\eadri\Dropbox\WorkCode\Github\Zou_lab_control_v2\packages\zlc_atom\src\zlc_atom\__init__.py


# zlc_atom: observation and orchestration

The two paths below share the same virtual devices and runtime plane. A camera measurement only observes frames; the experimenter owns `sequencer.load/fire` in the manual path, while `CalibrationTask` owns the complete pulse-to-report workflow in the automated path.

In [2]:
from pathlib import Path

import numpy as np
from zlc_durable import atomic_write_bytes
from zlc_runtime import SignalDataPlane

import zlc_atom
from zlc_atom.install import create_installation
from zlc_atom.nodes import calibration_pulse_template_bytes
from zlc_atom.nodes.calibration.pulse import (
    arm_sequencer, load_calibration_pulse_template, resolve_pulse,
)
from zlc_atom.nodes.camera_measurement import CameraMeasurementNode, CameraMeasurementRequest
from zlc_atom.nodes.calibration import CalibrationRequest, CalibrationTask, ReadoutModelKind
from zlc_atom.nodes.occupancy import OccupancyProcessor
from zlc_atom.nodes.calibration.calibration import FrameContract, calibrate

PACKAGE_ROOT = Path(zlc_atom.__file__).resolve().parents[2]
WORKSPACE = PACKAGE_ROOT / 'notebooks' / 'workspace'
PULSE_ROOT = WORKSPACE / 'pulses'
ARTIFACT_ROOT = WORKSPACE / 'data'
PULSE_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
template_path = PULSE_ROOT / 'imaging_template.json'
if not template_path.exists():
    atomic_write_bytes(template_path, calibration_pulse_template_bytes())
pulse_sequence = load_calibration_pulse_template(template_path)
package_identity = {'package': zlc_atom.__name__, 'version': zlc_atom.__version__}
package_identity

{'package': 'zlc_atom', 'version': '0.1.0'}

## 1. Manual path: the user owns the sequencer

The measurement is armed and collects exact frames. The pulse is resolved and fired by the caller, outside `CameraMeasurementNode`.

In [3]:
installation = create_installation('virtual')
plane = SignalDataPlane()
camera = installation.device('camera')
sequencer = installation.device('sequencer')

manual_pulse = resolve_pulse(
    pulse_sequence,
    path=template_path,
    board=sequencer.describe(),
    api_values={
        'reference_probe_duration_before': 0.02,
        'readout_probe_duration': 0.005,
        'reference_probe_duration_after': 0.02,
    },
)
manual_measurement = CameraMeasurementNode(
    camera=camera,
    request=CameraMeasurementRequest('camera', 0.02, None, 1, 3),
    signal_plane=plane,
    producer='manual_measurement',
)
manual_capture = manual_measurement.prepare()

# Explicit experimenter-owned excitation: measurement never calls these methods.
arm_sequencer(sequencer, manual_pulse)
sequencer.fire()
sequencer.wait_done(1.0)
manual_result = manual_capture.collect()
manual_signal_keys = tuple(
    manual_measurement.signal_key(f'frame_{index}') for index in range(3)
)
manual_signal = manual_result.publication.value(manual_signal_keys[0])
manual_observation = {
    'frames': len(manual_result.frames),
    'same_publication_siblings': tuple(manual_result.publication.signals),
    'values_shape': manual_signal.values.shape,
    'dtype': manual_result.frames[0].image.dtype.str,
}
manual_observation

{'frames': 3,
 'same_publication_siblings': ('@logic/manual_measurement/frame_0',
  '@logic/manual_measurement/frame_1',
  '@logic/manual_measurement/frame_2'),
 'values_shape': (1, 1, 96, 128),
 'dtype': '<u2'}

In [4]:
manual_monitor_node = CameraMeasurementNode(
    camera=camera,
    request=CameraMeasurementRequest('camera', 0.02, None, 0, 1),
    signal_plane=plane,
    producer='manual_monitor',
)
manual_monitor = manual_monitor_node.monitor(buffer_frames=1)
arm_sequencer(sequencer, manual_pulse)
sequencer.fire()
sequencer.wait_done(1.0)
monitor_record = manual_monitor.poll()
monitor_front = plane.freeze()
manual_monitor_observation = (
    monitor_record.image.shape,
    monitor_record.image.dtype.str,
    manual_monitor_node.signal_key('frame_0') in monitor_front.signals,
)
manual_monitor.close()
manual_monitor_observation

((96, 128), '<u2', True)

## 2. Automated path: the task owns the whole experiment

The task consumes the project `pulses/imaging_template.json` long-short-long bracket at Start, loads/fires the sequencer for each repeat, and computes one result from the long reference consensus and short readout. It saves that result once as JSON and passes the same object to `zlc_plot` for six PNG report images. A hosted task publishes only the current capture preview while the loop is running; Workbench does not render or open the report.

In [5]:
calibration_request = CalibrationRequest(
    camera_key='camera', sequencer_key='sequencer',
    pulse_template='imaging_template.json', repeats=30,
    reference_exposure_seconds=0.02, readout_exposure_seconds=0.005,
    roi_xywh=None, default_model_kind=ReadoutModelKind.BOX,
    threshold_method='empirical', box_half_width=1, box_reducer='mean',
    psf_half_width=3, psf_padding=3, detection_spot_sigma=1.0,
    detection_min_distance=3, detection_sigma=6.0,
)
task_result = CalibrationTask(
    camera=camera, sequencer=sequencer, request=calibration_request,
    pulse_sequence=pulse_sequence, pulse_path=template_path,
    artifact_directory=ARTIFACT_ROOT,
).run()

occupancy_node = OccupancyProcessor(task_result.calibration, signal_plane=plane)
occupancy_result = occupancy_node.process(task_result.short, generation='notebook', revision=1)
report_root = task_result.artifact_path.with_suffix('') / 'report'
auto_observation = {
    'reference_pairs': len(task_result.reference),
    'short_frames': len(task_result.short),
    'artifact_path': str(task_result.artifact_path),
    'report_images': tuple(path.name for path in sorted(report_root.glob('*.png'))),
    'counts_shape': occupancy_result.counts.shape,
    'rate_shape': occupancy_result.rate.shape,
    'rate_mean': float(np.mean(occupancy_result.rate)),
}
auto_observation

{'reference_pairs': 30,
 'short_frames': 30,
 'artifact_path': 'C:\\Users\\eadri\\Dropbox\\WorkCode\\Github\\Zou_lab_control_v2\\packages\\zlc_atom\\notebooks\\workspace\\data\\calibration-2.json',
 'report_images': ('box.png',
  'fidelity.png',
  'psf.png',
  'psf_kernels.png',
  'site_map.png',
  'uniform_psf.png'),
 'counts_shape': (30, 35),
 'rate_shape': (30,),
 'rate_mean': 0.4819047619047619}

In [6]:
# The frozen mathematical oracle remains independent of the virtual device path.
with np.load(PACKAGE_ROOT / 'tests/fixtures/main_readout_oracle.npz', allow_pickle=False) as oracle:
    oracle_result = calibrate(
        oracle['input_reference_frames'],
        oracle['input_short_frames'],
        frame_contract=FrameContract((34, 40), exposure_seconds=0.005),
    )
    box_predictions = oracle_result.report['models']['box']['predictions']
    oracle_errors = int(np.count_nonzero(box_predictions != oracle['input_latent_occupancy']))
oracle_errors

29

In [7]:
plane.close()
installation.close()